# Lab 7 — اجعل Bayan أسرع وشغّله كخدمة
**SDA-AIE-211 **

## الهدف
سنقوم بأربع خطوات فقط:
1. نقيس سرعة نموذج التصنيف.
2. نحوله إلى ONNX ثم INT8 ونقارن السرعة والنتائج.
3. نشغّل النموذج داخل API.
4. نختبر الخدمة ونوثّق النتائج.

**قبل البدء:** أكمل Lab 3a، وتأكد من وجود:
- `bayan_topic_model`
- `data/bench_mix.npy`

> شغّل الخلايا بالترتيب. لا تكتب أن الهدف تحقق إلا إذا أظهرت نتائج تشغيلك ذلك.

## 1. تثبيت المكتبات وتحميل النموذج

In [11]:
import zipfile

zip_path = "/content/bayan_topic_model-20261110T104342Z-1-001.zip"

if zipfile.is_zipfile(zip_path):
    with zipfile.ZipFile(zip_path) as z:
        z.extractall('/content')
    print('تم فك الضغط ✅')
else:
    print('ملف ZIP غير صالح؛ أعد تنزيله ورفعه بعد اكتمال التنزيل')

تم فك الضغط ✅


In [13]:
# تثبيت المكتبات المطلوبة (يكفي تشغيله مرة واحدة، ويمكنك وضع # قبله بعد ذلك)
# optimum حُذفت لأنها تتعارض مع إصدار transformers الحديث في Colab؛ نصدّر ONNX بـ torch مباشرة
%pip -q install transformers onnx onnxscript onnxruntime fastapi uvicorn httpx torch numpy

# المكتبات: نكتب فائدة كل مكتبة بجانب اسمها
from pathlib import Path                         # التعامل مع مسارات الملفات
import time                                      # قياس زمن التنفيذ
import re                                        # تنظيف النصوص
import shutil                                    # العثور على أمر تصدير ONNX
import subprocess                                # تشغيل أمر التصدير
import threading                                 # تشغيل API في الخلفية
import numpy as np                               # حساب p50 و p99 ومقارنة النتائج
import torch                                     # تشغيل نموذج PyTorch الأصلي
import onnxruntime as ort                        # تشغيل نموذج ONNX
import httpx                                     # إرسال طلبات HTTP إلى API
import uvicorn                                   # تشغيل خادم FastAPI
from transformers import AutoTokenizer, AutoModelForSequenceClassification  # تحميل tokenizer ونموذج التصنيف
from onnxruntime.quantization import quantize_dynamic, QuantType             # ضغط نموذج ONNX إلى INT8
from fastapi import FastAPI                      # إنشاء API
from pydantic import BaseModel, Field            # التحقق من مدخلات API

# مجلد المشروع على جهازك (غيّره إذا نقلت الملفات)
BASE_DIR = Path("")
print('مجلد العمل | working folder:', BASE_DIR.resolve())
assert BASE_DIR.exists(), f'المجلد غير موجود: {BASE_DIR}'

# البحث عن النموذج وملف الاختبار داخل المجلد وفروعه
def find_first(name, folder_only=False):
    for p in BASE_DIR.rglob(name):
        if p.is_dir() or not folder_only:
            return p
    return None

MODEL_DIR = find_first('bayan_topic_model', folder_only=True)
BENCH_FILE = find_first('bench_mix.npy')
inside = sorted(p.name for p in BASE_DIR.iterdir())   # محتويات المجلد للمساعدة عند الخطأ

assert MODEL_DIR is not None, f'مجلد bayan_topic_model غير موجود. محتويات المجلد: {inside}'
assert (MODEL_DIR / 'config.json').exists(), f'المجلد {MODEL_DIR} لا يحتوي config.json (ليس نموذج transformers)'
assert BENCH_FILE is not None, f'ملف bench_mix.npy غير موجود. محتويات المجلد: {inside}'
DATA_DIR = BENCH_FILE.parent
print('النموذج | model :', MODEL_DIR)
print('الاختبار | bench:', BENCH_FILE)

# تحميل رسائل الاختبار
texts = np.load(BENCH_FILE, allow_pickle=True).tolist()

# تحميل النموذج الأصلي
tokenizer = AutoTokenizer.from_pretrained(MODEL_DIR)
model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR).eval()
torch.set_num_threads(4)  # استخدام 4 أنوية CPU

print('عدد رسائل الاختبار:', len(texts))
print('تم تحميل النموذج ✅')

مجلد العمل | working folder: /content
النموذج | model : bayan_topic_model
الاختبار | bench: bench_mix.npy


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

عدد رسائل الاختبار: 2000
تم تحميل النموذج ✅


In [3]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## 2. قياس سرعة النموذج الأصلي

سنقارن إعدادين:
- `FP32 512`: طول 512 مع Padding ثابت.
- `FP32 128`: طول 128 مع Padding حسب طول النص.

**p50** هو الزمن الوسيط، و**p99** يوضح زمن أبطأ 1% من الطلبات.

In [14]:
def predict(text, max_length, fixed_padding):
    # تحويل النص إلى أرقام ثم تشغيل النموذج
    inputs = tokenizer(
        str(text),
        max_length=max_length,
        truncation=True,
        padding='max_length' if fixed_padding else True,
        return_tensors='pt'
    )
    with torch.inference_mode():                 # لا نحتاج حسابات التدريب
        output = model(**inputs)
    return int(output.logits.argmax(-1).item())  # إرجاع رقم الفئة المتوقعة


def benchmark(function):
    times = []
    for i in range(100):                         # نكرر القياس 100 مرة
        start = time.perf_counter()
        function(texts[i % len(texts)])
        times.append((time.perf_counter() - start) * 1000)
    return {
        'p50_ms': round(float(np.percentile(times, 50)), 2),
        'p99_ms': round(float(np.percentile(times, 99)), 2)
    }


# قياس الإعدادين
fp32_512 = benchmark(lambda x: predict(x, 512, True))
fp32_128 = benchmark(lambda x: predict(x, 128, False))

print('FP32 512:', fp32_512)
print('FP32 128:', fp32_128)

FP32 512: {'p50_ms': 1005.17, 'p99_ms': 2512.99}
FP32 128: {'p50_ms': 60.67, 'p99_ms': 239.35}


## 3. تصدير النموذج إلى ONNX ثم INT8

In [15]:
# إنشاء مجلد لحفظ نموذج ONNX
onnx_dir = BASE_DIR / 'topic_onnx'
onnx_dir.mkdir(exist_ok=True)

# تصدير النموذج إلى ONNX باستخدام torch (بدون Optimum)
class LogitsOnly(torch.nn.Module):               # غلاف بسيط يعيد درجات الفئات (logits) فقط
    def __init__(self, base):
        super().__init__()
        self.base = base
    def forward(self, input_ids, attention_mask):
        return self.base(input_ids=input_ids, attention_mask=attention_mask).logits

export_model = AutoModelForSequenceClassification.from_pretrained(MODEL_DIR, attn_implementation='eager').eval()
dummy = tokenizer(['الخدمة لا تعمل'], max_length=128, truncation=True,
                  padding='max_length', return_tensors='pt')      # مثال واحد يحدد شكل المدخلات
onnx_file = onnx_dir / 'model.onnx'
export_args = dict(
    input_names=['input_ids', 'attention_mask'], output_names=['logits'],
    dynamic_axes={'input_ids': {0: 'batch', 1: 'sequence'},       # يقبل أي عدد رسائل وأي طول
                  'attention_mask': {0: 'batch', 1: 'sequence'},
                  'logits': {0: 'batch'}},
    opset_version=17
)
with torch.inference_mode():
    try:
        torch.onnx.export(LogitsOnly(export_model), (dummy['input_ids'], dummy['attention_mask']),
                          str(onnx_file), dynamo=False, **export_args)      # الطريقة المستقرة
    except Exception as exc:
        print('الطريقة الأولى فشلت، نجرب الثانية:', str(exc)[:200])
        torch.onnx.export(LogitsOnly(export_model), (dummy['input_ids'], dummy['attention_mask']),
                          str(onnx_file), dynamo=True, **export_args)       # الطريقة الجديدة (تحتاج onnxscript)
assert onnx_file.exists(), 'لم يُنشأ ملف ONNX'

# إنشاء نسخة INT8 من نموذج ONNX
int8_file = str(BASE_DIR / 'topic_int8.onnx')
quantize_dynamic(str(onnx_file), int8_file, weight_type=QuantType.QInt8) #make int8 copy

# فتح النموذجين باستخدام ONNX Runtime
onnx_model = ort.InferenceSession(str(onnx_file), providers=['CPUExecutionProvider'])
int8_model = ort.InferenceSession(int8_file, providers=['CPUExecutionProvider'])
print('تم إنشاء نسختي ONNX و INT8 ✅')

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

/tmp/ipykernel_568/2102220508.py:26: DeprecationWarning: You are using the legacy TorchScript-based ONNX export. Starting in PyTorch 2.9, the new torch.export-based ONNX exporter has become the default. Learn more about the new export logic: https://docs.pytorch.org/docs/stable/onnx_export.html. For exporting control flow: https://pytorch.org/tutorials/beginner/onnx/export_control_flow_model_to_onnx_tutorial.html
  torch.onnx.export(LogitsOnly(export_model), (dummy['input_ids'], dummy['attention_mask']),
/usr/local/lib/python3.13/dist-packages/transformers/masking_utils.py:208: TracerWarning: Converting a tensor to a Python boolean might cause the trace to be incorrect. We can't record the data flow of Python values, so this value will be treated as a constant in the future. This means that the trace might not generalize to other inputs!
  if (padding_length := kv_length + kv_offset - attention_mask.shape[-1]) > 0:
/usr/local/lib/python3.13/dist-packages/torch/onnx/_internal/torchscrip

تم إنشاء نسختي ONNX و INT8 ✅


## 4. قياس السرعة ومقارنة التوقعات

In [16]:
def predict_onnx(text, session):
    # تجهيز النص بالطريقة نفسها ثم تشغيل نموذج ONNX
    inputs = tokenizer(str(text), max_length=128, truncation=True,
                       padding='max_length', return_tensors='np')
    # نرسل المدخلات التي يحتاجها النموذج فقط
    names = [item.name for item in session.get_inputs()]
    inputs = {name: inputs[name] for name in names}
    output = session.run(None, inputs)[0]
    return int(np.argmax(output, axis=1)[0])


# قياس سرعة ONNX و INT8
onnx_speed = benchmark(lambda x: predict_onnx(x, onnx_model))
int8_speed = benchmark(lambda x: predict_onnx(x, int8_model))
print('ONNX FP32:', onnx_speed)
print('ONNX INT8:', int8_speed)

# مقارنة نتائج INT8 بنتائج النموذج الأصلي على 100 رسالة
sample = texts[:100]
original = [predict(x, 128, False) for x in sample] #original model
compressed = [predict_onnx(x, int8_model) for x in sample] #int8 model
agreement = np.mean(np.array(original) == np.array(compressed)) #إذا كان التوقعان متطابقين نحصل على True (1) حساب نسبة التطابق
print('Prediction agreement:', round(float(agreement), 3))

ONNX FP32: {'p50_ms': 110.18, 'p99_ms': 187.53}
ONNX INT8: {'p50_ms': 87.92, 'p99_ms': 137.65}
Prediction agreement: 0.97


## 5. اختيار النسخة المناسبة

نختار INT8 إذا كانت نتائجه قريبة من النموذج الأصلي. في هذا التمرين نستخدم **97% تطابق** كحد إرشادي، وليس كمعيار عام لكل المشاريع.

نحتفظ دائمًا بنسخة ONNX FP32 كخطة رجوع (Rollback).

In [17]:
# اختيار النسخة التي ستعمل داخل الخدمة
use_int8 = agreement >= 0.97
selected_model = int8_model if use_int8 else onnx_model

print('النسخة المختارة:', 'INT8' if use_int8 else 'ONNX FP32')
print('سبب الاختيار: مقارنة السرعة مع الحفاظ على تقارب التوقعات.')

النسخة المختارة: INT8
سبب الاختيار: مقارنة السرعة مع الحفاظ على تقارب التوقعات.


## 6. تجهيز التصنيف وفحوصات البداية (Canary)

الـ Canary اختبار صغير نتأكد به أن تنظيف النص وTokenizer يعملان قبل استقبال المستخدمين.

In [18]:
# ✍️ TODO 1: استخدم نفس آلية التنظيف التي استخدمتها في Lab 1
# 1) افتح Lab 1 وانسخ الدوال الثلاث (mask_pii و normalize و preprocess) من خلية PREPROCESSING_VERSION.
# 2) الصقها هنا فوق الدالة clean_text مباشرة.
# 3) غيّر جسم الدالة clean_text ليصبح:   return preprocess(text)
# لماذا؟ يجب أن تنظّف الخدمة النص بالطريقة نفسها التي دُرّب بها النموذج، وتُخفي الهاتف والبريد (PII).

# Version of the preprocessing pipeline

import re
import unicodedata

PREPROCESSING_VERSION = "bayan-preproc-v1"


# Function to hide personal information (PII)
def mask_pii(text):

    text = str(text)

    # Regular Expression for email addresses:
    # [A-Za-z0-9._%+-]+ = characters before @
    # @ = the @ symbol
    # [A-Za-z0-9.-]+ = email domain
    # \. = dot before the extension
    # [A-Za-z]{2,} = extension with at least 2 letters
    text = re.sub(
        r'[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Za-z]{2,}',
        '[EMAIL]', text
    )

    # Regular Expression for Saudi mobile numbers:
    # 05 = Saudi mobile prefix
    # (?:[\s-]?\d){8} = exactly 8 digits, each may have a space or hyphen before it
    #                   (so 055-123-4567 and 055 123 4567 are both matched)
    # (?<!\d) and (?!\d) prevent matching part of a longer number

    #TODO: Add regular expression for Saudi mobile numbers
    text = re.sub(
        r'(?<!\d)05(?:[\s-]?\d){8}(?!\d)',
        '[PHONE]',
        text
    )


    # Regular Expression for Saudi national ID:
    # [12] = starts with 1 or 2
    # \d{9} = followed by 9 digits
    # (?<!\d) and (?!\d) prevent matching part of a longer number

    #TODO: Add regular expression for Saudi national ID
    text = re.sub(
        r'(?<!\d)[12]\d{9}(?!\d)',
        '[ID]',
        text
    )

    return text


# Function to normalize Arabic and English text
def normalize(text):

    # Convert text to Unicode NFC format
    text = unicodedata.normalize('NFC', str(text))

    # Remove Arabic Tatweel (ـ)
    text = text.replace('ـ', '')

    # Replace multiple spaces with one space
    text = re.sub(r'\s+', ' ', text)

    # Remove spaces at the beginning and end
    return text.strip()


# Function to apply preprocessing in the correct order
def preprocess(text):

    # Step 1: Hide personal information FIRST
    text = mask_pii(text)

    # Step 2: Normalize the text
    text = normalize(text)

    return text

def clean_text(text):
    return preprocess(text)


def classify(text):
    clean = clean_text(text)
    pred_id = predict_onnx(clean, selected_model)
    label = model.config.id2label.get(pred_id, str(pred_id))
    return {'text': clean, 'topic': label, 'model_version': 'bayan-topic-v1'}


# اختبارات بداية بسيطة
assert clean_text('  testـ  text  ') == 'test text'
assert len(tokenizer('الخدمة لا تعمل في الرياض')['input_ids']) > 2
print('Canary checks: GREEN ✅')

# فحص TODO 1: هل أخفت الدالة رقم الهاتف؟
if '[PHONE]' in clean_text('اتصلوا بي على 0551234567'):
    print('TODO 1 ✅ التنظيف مطابق لـ Lab 1')
else:
    print('TODO 1 ⏳ لم تُكمل بعد: رقم الهاتف لا يزال ظاهرًا في النص')
print(classify('تم خصم المبلغ مرتين من الفاتورة'))

Canary checks: GREEN ✅
TODO 1 ✅ التنظيف مطابق لـ Lab 1
{'text': 'تم خصم المبلغ مرتين من الفاتورة', 'topic': 'billing', 'model_version': 'bayan-topic-v1'}


## 7. إنشاء API

Use FastAPI to build one integrated API that connects the Classification, NER, and Semantic Search components to process complaints and return their results.

In [20]:
%pip -q install sentence-transformers faiss-cpu

import json
import pandas as pd
import faiss
from sentence_transformers import SentenceTransformer
from transformers import pipeline

CASES_FILE = find_first("bayan_cases.csv")
INDEX_FILE = find_first("bayan_cases.faiss")
MANIFEST_FILE = find_first("bayan_search_manifest.json")
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")
NER_DIR = Path("/content/drive/MyDrive/bayan_ner_model")

assert CASES_FILE is not None, "ارفع bayan_cases.csv إلى /content"
assert NER_DIR is not None, "فك ضغط bayan_ner_model داخل /content"

cases = pd.read_csv(CASES_FILE)

if INDEX_FILE is not None and MANIFEST_FILE is not None:
    with MANIFEST_FILE.open(encoding="utf-8") as f:
        embedding_model = json.load(f)["model"]
    embedder = SentenceTransformer(embedding_model)
    faiss_index = faiss.read_index(str(INDEX_FILE))
else:
    embedding_model = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    embedder = SentenceTransformer(embedding_model)

    columns = ["title", "description", "resolution"]
    case_texts = cases[columns].fillna("").astype(str).agg(" ".join, axis=1)
    vectors = embedder.encode(
        [clean_text(text) for text in case_texts],
        batch_size=64,
        show_progress_bar=True,
        normalize_embeddings=True,
        convert_to_numpy=True,
    ).astype("float32")

    faiss_index = faiss.IndexFlatIP(vectors.shape[1])
    faiss_index.add(vectors)

assert faiss_index.ntotal == len(cases), "فهرس FAISS لا يطابق عدد الحالات"

ner_pipe = pipeline(
    "token-classification",
    model=str(NER_DIR),
    aggregation_strategy="simple",
)

print("Semantic Search ✅", len(cases))
print("NER ✅", NER_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


modules.json:   0%|          | 0.00/229 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/3.89k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/645 [00:00<?, ?B/s]

... (7 outputs left out)

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Batches:   0%|          | 0/313 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/199 [00:03<?, ?it/s]

Semantic Search ✅ 20000
NER ✅ /content/drive/MyDrive/bayan_ner_model


In [21]:
from fastapi import FastAPI
from pydantic import BaseModel, Field

app = FastAPI(title="Bayan API")


class ComplaintRequest(BaseModel):
    text: str = Field(..., min_length=1)


class BatchComplaintRequest(BaseModel):
    complaints: list[ComplaintRequest] = Field(..., min_length=1, max_length=10)


@app.get("/health")
def health():
    return {"status": "ok"}


@app.post("/v1/analyze")
def analyze_complaint(request: ComplaintRequest):
    classification = classify(request.text)
    text = classification["text"]

    entities = [
        {
            "text": item["word"],
            "type": item["entity_group"],
            "score": round(float(item["score"]), 3),
        }
        for item in ner_pipe(text)
    ]

    vector = embedder.encode(
        [text], normalize_embeddings=True, convert_to_numpy=True
    ).astype("float32")
    scores, positions = faiss_index.search(vector, 3)

    similar_cases = []
    for score, position in zip(scores[0], positions[0]):
        if position >= 0:
            row = cases.iloc[int(position)]
            similar_cases.append({
                "case_id": str(row["case_id"]),
                "title": clean_text(row["title"]),
                "similarity": round(float(score), 3),
            })

    return {
        "input_text": text,
        "classification": classification,
        "entities": entities,
        "similar_cases": similar_cases,
    }


@app.post("/v1/analyze-batch")
def analyze_batch(request: BatchComplaintRequest):
    return {
        "results": [
            analyze_complaint(complaint)
            for complaint in request.complaints
        ]
    }

In [22]:
from fastapi.testclient import TestClient
from pprint import pprint

arabic = {"text": "أبلغت نورة عن انقطاع المياه في جدة. اتصلوا على 0551234567"}
english = {"text": "I was charged twice for the same bill."}

with TestClient(app) as client:
    single = client.post("/v1/analyze", json=arabic)
    single.raise_for_status()
    assert "[PHONE]" in single.json()["input_text"]
    assert "0551234567" not in single.json()["input_text"]
    print("Single complaint ✅")
    pprint(single.json())

    batch = client.post(
        "/v1/analyze-batch",
        json={"complaints": [arabic, english]},
    )
    batch.raise_for_status()
    assert len(batch.json()["results"]) == 2
    print("Batch analysis ✅")
    pprint(batch.json())

Single complaint ✅
{'classification': {'model_version': 'bayan-topic-v1',
                    'text': 'أبلغت نورة عن انقطاع المياه في جدة. اتصلوا على '
                            '[PHONE]',
                    'topic': 'outage'},
 'entities': [{'score': 0.997, 'text': 'نور', 'type': 'PERSON'},
              {'score': 0.974, 'text': 'ة', 'type': 'PERSON'},
              {'score': 0.996, 'text': 'ان', 'type': 'SERVICE'},
              {'score': 0.904, 'text': 'قطاع المياه', 'type': 'SERVICE'},
              {'score': 0.998, 'text': 'جد', 'type': 'LOCATION'},
              {'score': 0.997, 'text': 'ة', 'type': 'LOCATION'},
              {'score': 0.878, 'text': '[', 'type': 'LOCATION'},
              {'score': 0.329, 'text': 'PHONE', 'type': 'LOCATION'}],
 'input_text': 'أبلغت نورة عن انقطاع المياه في جدة. اتصلوا على [PHONE]',
 'similar_cases': [{'case_id': 'CASE11984',
                    'similarity': 0.552,
                    'title': 'outage'},
                   {'case_id': 'CASE17

In [23]:
import threading
import time

import httpx
import uvicorn
from google.colab import output

server = uvicorn.Server(
    uvicorn.Config(app, host="127.0.0.1", port=8000, log_level="warning")
)
threading.Thread(target=server.run, daemon=True).start()

for _ in range(50):
    if server.started:
        break
    time.sleep(0.2)

assert server.started, "الخادم ما بدأ. تأكد أن المنفذ 8000 غير مستخدم."
response = httpx.get("http://127.0.0.1:8000/health", timeout=5)
response.raise_for_status()
print(response.json())

output.serve_kernel_port_as_iframe(8000, path="/docs", height=700)

{'status': 'ok'}


<IPython.core.display.Javascript object>

In [24]:
import json
import shutil
from pathlib import Path

search_dir = Path("/content/drive/MyDrive/bayan_search")
search_dir.mkdir(parents=True, exist_ok=True)

faiss.write_index(faiss_index, str(search_dir / "bayan_cases.faiss"))
shutil.copy2(CASES_FILE, search_dir / "bayan_cases.csv")
(search_dir / "bayan_search_manifest.json").write_text(
    json.dumps({"model": embedding_model}, indent=2),
    encoding="utf-8",
)

print("Search saved:", search_dir)

Search saved: /content/drive/MyDrive/bayan_search


In [25]:
import asyncio
from time import perf_counter

import httpx
import numpy as np

URL = "http://127.0.0.1:8000/v1/analyze"
USERS = 16
REQUESTS_PER_USER = 10

samples = [
    {"text": "أبلغت نورة عن انقطاع المياه في جدة."},
    {"text": "I was charged twice for the same bill."},
]

async def run_benchmark():
    limits = httpx.Limits(
        max_connections=USERS,
        max_keepalive_connections=USERS,
    )

    async with httpx.AsyncClient(timeout=60, limits=limits) as client:
        # إحماء الخدمة قبل بدء القياس
        for sample in samples:
            response = await client.post(URL, json=sample)
            response.raise_for_status()

        async def run_user(user_id):
            results = []
            for i in range(REQUESTS_PER_USER):
                sample = samples[(user_id + i) % len(samples)]
                start = perf_counter()
                try:
                    response = await client.post(URL, json=sample)
                    status = response.status_code
                except httpx.RequestError as exc:
                    status = type(exc).__name__
                results.append((status, (perf_counter() - start) * 1000))
            return results

        runs = await asyncio.gather(*(run_user(i) for i in range(USERS)))

    results = [item for run in runs for item in run]
    successful = [ms for status, ms in results if status == 200]
    failures = [status for status, _ in results if status != 200]

    print(f"Concurrent users: {USERS}")
    print(f"HTTP 200: {len(successful)}/{len(results)}")
    print(f"Failures: {len(failures)}")
    if failures:
        print("Failure types:", sorted(set(map(str, failures))))
    if successful:
        print(f"p50: {np.percentile(successful, 50):.2f} ms")
        print(f"p99: {np.percentile(successful, 99):.2f} ms")

await run_benchmark()

Concurrent users: 16
HTTP 200: 160/160
Failures: 0
p50: 3893.55 ms
p99: 6310.45 ms
